<a href="https://colab.research.google.com/github/jaalvalcan/GEE_index_sets/blob/main/Netcdf_Niger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. SETUP ENVIRONMENTS & STREAM GEODATA
# ==========================================
!pip install netcdf4 geopandas -q

import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Polygon

# Stream low-res world borders directly from a public repository into memory
print("Fetching geographical base layers...")
countries_url = "https://raw.githubusercontent.com/vasturiano/globe.gl/master/example/datasets/ne_110m_admin_0_countries.geojson"
world = gpd.read_file(countries_url)

# ==========================================
# 2. FETCH & PROCESS NETCDF TEMPERATURE DATA
# ==========================================
print("Connecting to NOAA NetCDF Server...")
opendap_url = "https://psl.noaa.gov/thredds/dodsC/Datasets/ncep.reanalysis.derived/surface/air.mon.mean.nc"
ds = xr.open_dataset(opendap_url)

# Convert 0-360° Longitudes to standard -180° to 180° mapping coordinates
ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180)).sortby(['lon', 'lat'])

# Extract your exact 5-vertex polygon box for the last 10 years
lon_min, lon_max = -9.887695, 7.646484
lat_min, lat_max = 8.711359, 17.518344

grid_subset = ds['air'].sel(
    time=slice("2016-01-01", "2026-03-01"),
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

# FIXED: Removed the '- 273.15' adjustment because this specific dataset natively reads in Celsius (°C)
mean_grid = grid_subset.mean(dim='time')

# ==========================================
# 3. DEFINE NIGER RIVER PATH GRAPHICALLY
# ==========================================
# Coordinate checkpoints tracking the actual physical path of the Niger River loop
niger_river_lon = [-10.0, -8.0, -6.2, -4.2, -3.0, -1.6, 0.0, 1.3, 2.1, 3.5, 4.5, 6.3, 7.0]
niger_river_lat = [10.0, 12.6, 13.4, 14.5, 16.7, 16.8, 16.3, 15.0, 13.5, 11.9, 11.3, 8.0, 6.0]

# ==========================================
# 4. BUILD THE 3-PANEL GRAPHICS CANVAS
# ==========================================
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 7), gridspec_kw={'width_ratios': [1, 1.2, 1]})

# ------------------------------------------
# PANEL 1: MACRO-GEOGRAPHICAL CONTEXT
# ------------------------------------------
# Crop world map down to a wide West Africa viewpoint
west_africa = world.cx[-18:15, 4:24]
west_africa.plot(ax=ax1, facecolor='#f4f4f0', edgecolor='#999999', linewidth=0.8)

# Overlay a highlighted bounding box representing your Area of Interest (AOI)
aoi_box = plt.Rectangle((lon_min, lat_min), lon_max - lon_min, lat_max - lat_min,
                        edgecolor='crimson', facecolor='none', linewidth=2.5, linestyle='--', label='Your AOI Box')
ax1.add_patch(aoi_box)

# Label key geographic reference regions
ax1.text(-4, 19, "SAHARA DESERT", fontsize=9, color='gray', style='italic', weight='bold')
ax1.text(-2, 6, "GULF OF GUINEA", fontsize=9, color='blue', style='italic', weight='bold')
ax1.text(-4, 12, "AOI", color='crimson', weight='bold', fontsize=12)

ax1.set_title("1. Regional Macro-Context\n(West Africa)", fontsize=13, weight='bold')
ax1.set_xlim(-16, 12)
ax1.set_ylim(5, 23)
ax1.grid(True, linestyle=':', alpha=0.5)
ax1.legend(loc='upper right')

# ------------------------------------------
# PANEL 2: ZOOMED GRID WITH BORDER OVERLAYS
# ------------------------------------------
# Render the NetCDF pixel arrays as the underlying thermal layer
mesh = ax2.pcolormesh(mean_grid.lon, mean_grid.lat, mean_grid, cmap='YlOrRd', shading='auto', alpha=0.85)

# Overlay thin political borders on top of the temperature grid pixels
world.plot(ax=ax2, facecolor='none', edgecolor='#444444', linewidth=1.2)

# Draw the blue ribbon path of the Niger River
ax2.plot(niger_river_lon, niger_river_lat, color='#1111d6', linewidth=3.5, label='Niger River Course', zorder=4)

# Dynamic text mapping inside the micro-grid
ax2.text(-8, 14.0, "MALI\n(Arid North)", color='#7f1d1d', weight='bold', fontsize=10)
ax2.text(-1, 12.5, "BURKINA\nFASO", color='#14532d', weight='bold', fontsize=9)
ax2.text(5.0, 12.0, "NIGERIA\n(Wet Forest)", color='#14532d', weight='bold', fontsize=10)

ax2.set_title("2. Micro-Grid Overlay Map\n(NetCDF Cells + Borders + River)", fontsize=13, weight='bold')
ax2.set_xlim(lon_min, lon_max)
ax2.set_ylim(lat_min, lat_max)
cbar = fig.colorbar(mesh, ax=ax2, pad=0.03)
cbar.set_label("10-Year Mean Temp (°C)", fontsize=11)
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.2)

# ------------------------------------------
# PANEL 3: LATIDUDINAL CLIMATE PROFILE
# ------------------------------------------
# Flatten longitudes to map the pure South-to-North temperature path
lat_profile = mean_grid.mean(dim='lon')
ax3.plot(lat_profile.lat, lat_profile, marker='o', color='darkorange', linewidth=2, markersize=5)

# Highlight the river's regional boundary line around 13.5°N Latitude
ax3.axvline(x=13.5, color='#1d3557', linestyle='--', linewidth=2, label='Niger River Boundary Axis (~13.5°N)')

ax3.set_title("3. The Environmental Gradient\n(South-to-North Profile)", fontsize=13, weight='bold')
ax3.set_xlabel("Latitude (Degrees North)", fontsize=11)
ax3.set_ylabel("Mean Temperature (°C)", fontsize=11)
ax3.grid(True, linestyle=':', alpha=0.6)
ax3.legend(loc='lower right')

# Output display adjustments
plt.tight_layout()
plt.show()